In [ ]:
# Update files
import importlib
import config as cfg
import utils.io
import utils.metrics
import data.data_loader
import data.data_augmentation
import models.multi_layer_perceptron
import models.convolutional_neural_networks

importlib.reload(cfg)
importlib.reload(utils.io)
importlib.reload(utils.metrics)
importlib.reload(data.data_loader)
importlib.reload(data.data_augmentation)
importlib.reload(models.multi_layer_perceptron)
importlib.reload(models.convolutional_neural_networks)

# Import libraries  
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split

# Import functions
## Data
from data.data_loader import ButterflyDataset
from utils.io import print_images
from utils.metrics import analyze_df

## Metric
from utils.metrics import evaluate_network, save_acc_graph

## MLP
from models.multi_layer_perceptron import MLP, fit

## CNN
from models.convolutional_neural_networks import CNN, fit

# Choose device
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():    
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

## First, we read the dataset, preprocess the images and encapsulate them into dataloader form.

In [ ]:
# load the data
df = pd.read_csv(cfg.TRAIN_LABELS_PATH)

df_train, df_val = train_test_split(df, test_size=cfg.TEST_SIZE, random_state=42, stratify=df['label'])

# preprocessing
data_transform = transforms.Compose([
    transforms.Resize((cfg.IMAGE_SIZE, cfg.IMAGE_SIZE)),
    transforms.ToTensor()
])

train_dataset = ButterflyDataset(df=df_train, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)
val_dataset = ButterflyDataset(df=df_val, img_dir=cfg.TRAIN_IMG_DIR, transform=data_transform)

train_loader = data.DataLoader(train_dataset, batch_size=cfg.BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(val_dataset, batch_size=cfg.BATCH_SIZE, shuffle=False)
print("All images proccessed...")


In [ ]:
# Train data set analyze
print("#### TRAIN DATASET ####")
num_inputs, num_classes = analyze_df(df_train, train_dataset)
print_images(train_dataset, train_loader)

MLP

In [ ]:
dnn = MLP(input_size=num_inputs, hidden_sizes=cfg.MLP_HIDDEN_LAYER_SIZES, num_classes=num_classes)

# Escolher Loss Function
if cfg.MLP_LOSS_FUNCTION == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
elif cfg.MLP_LOSS_FUNCTION == "CrossEntropy_LabelSmoothing":
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.MLP_LABEL_SMOOTHING) 
elif cfg.MLP_LOSS_FUNCTION == "MultiMarginLoss":
    criterion = nn.MultiMarginLoss()
else:
    raise ValueError(f"Loss Function desconhecida no config: {cfg.MLP_LOSS_FUNCTION}")

# Escolher Otimizador
if cfg.MLP_OPTIM == "RMSprop":
    optimizer = optim.RMSprop(dnn.parameters(), lr=cfg.MLP_LR)
elif cfg.MLP_OPTIM == "ADAM":
    optimizer = optim.Adam(dnn.parameters(), lr=cfg.MLP_LR)
elif cfg.MLP_OPTIM == "ADAMW":
    optimizer = optim.AdamW(dnn.parameters(), lr=cfg.MLP_LR, weight_decay=cfg.MLP_WEIGHT_DECAY)
elif cfg.MLP_OPTIM == "SGD":
    optimizer = optim.SGD(dnn.parameters(), lr=cfg.MLP_LR, momentum=cfg.MLP_MOMENTUM)
else:
    raise ValueError(f"Otimizador desconhecido no config: {cfg.MLP_OPTIM}")

# Treino
print(f"A iniciar o treino com {cfg.MLP_OPTIM} e {cfg.MLP_LOSS_FUNCTION}...")
mlp_train_acc, mlp_val_acc, dnn = fit(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    nn=dnn,
    criterion=criterion, 
    optimizer=optimizer, 
    n_epochs=cfg.MLP_EPOCHS, 
    to_device=False
)

In [ ]:
print('Saving graphics showing loss results...')
save_acc_graph(mlp_train_acc, mlp_val_acc, cfg.mlp_results_path, cfg.mlp_results_folder_name, "RNN")

print('Evaluating with the training data...')
evaluate_network(
    net=dnn, 
    dataloader=train_loader, 
    device="cpu",
    save_path=cfg.mlp_results_path, 
    split_name="Treino"
)

print('Evaluating with the valid data...')
evaluate_network(
    net=dnn, 
    dataloader=val_loader, 
    device="cpu", 
    save_path=cfg.mlp_results_path, 
    split_name="Validacao"
)

CNN

In [ ]:
cnn = CNN(input_channels=3, num_classes=num_classes)

# Escolher Loss Function
if cfg.CNN_LOSS_FUNCTION == "CrossEntropy":
    criterion = nn.CrossEntropyLoss()
elif cfg.CNN_LOSS_FUNCTION == "CrossEntropy_LabelSmoothing":
    criterion = nn.CrossEntropyLoss(label_smoothing=cfg.CNN_LABEL_SMOOTHING) 
elif cfg.CNN_LOSS_FUNCTION == "MultiMarginLoss":
    criterion = nn.MultiMarginLoss()
else:
    raise ValueError(f"Loss Function desconhecida no config: {cfg.CNN_LOSS_FUNCTION}")

# Escolher Otimizador
if cfg.CNN_OPTIM == "RMSprop":
    optimizer = optim.RMSprop(cnn.parameters(), lr=cfg.CNN_LR)
elif cfg.CNN_OPTIM == "ADAM":
    optimizer = optim.Adam(cnn.parameters(), lr=cfg.CNN_LR)
elif cfg.CNN_OPTIM == "ADAMW":
    optimizer = optim.AdamW(cnn.parameters(), lr=cfg.CNN_LR, weight_decay=cfg.CNN_WEIGHT_DECAY)
elif cfg.CNN_OPTIM == "SGD":
    optimizer = optim.SGD(cnn.parameters(), lr=cfg.CNN_LR, momentum=cfg.CNN_MOMENTUM)
else:
    raise ValueError(f"Otimizador desconhecido no config: {cfg.CNN_OPTIM}")

# Treino
print(f"\nA iniciar o treino da CNN com {cfg.CNN_OPTIM} e {cfg.CNN_LOSS_FUNCTION}...")
cnn_train_acc, cnn_val_acc, cnn = fit(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    nn=cnn,
    criterion=criterion, 
    optimizer=optimizer, 
    n_epochs=cfg.CNN_EPOCHS, 
    to_device=False,
    device=device
)

In [ ]:
print('Saving graphics showing loss results...')
save_acc_graph(cnn_train_acc, cnn_val_acc, cfg.cnn_results_path, cfg.cnn_results_folder_name, "CNN")

print('Evaluating with the training data...')
evaluate_network(
    net=cnn, 
    dataloader=train_loader, 
    device="cpu",
    save_path=cfg.cnn_results_path, 
    split_name="Treino",
    is_cnn = "True"
)

print('Evaluating with the valid data...')
evaluate_network(
    net=cnn, 
    dataloader=val_loader, 
    device="cpu", 
    save_path=cfg.cnn_results_path, 
    split_name="Validacao",
    is_cnn = "True"
)